# Download JazzStandards Dataset from iRealPro

This notebook is to transform the JazzStandards JSON dataset into a Pandas dataframe.

Dataset retrieved from: https://github.com/mikeoliphant/JazzStandards/tree/main

In [1]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 120)

RAW_JSON_PATH = Path("data/raw/irealpro/JazzStandards.json")
#RAW_PANDAS_PATH = Path("data\raw\irealpro\JazzStandards.parquet")

In [2]:
from pathlib import Path

print(Path.cwd())

c:\Users\mafal\Documents\MSc Data Science\2nd Year\Thesis\Practice\GenChordProg\data\raw\irealpro


In [ ]:
RAW_JSON_PATH = Path("C:\\Users\\mafal\\Documents\\MSc Data Science\\2nd Year\\Thesis\\Practice\\ChordProgRLHF\\data\\raw\\irealpro\\JazzStandards.json")

print(RAW_JSON_PATH.exists())
print(RAW_JSON_PATH.resolve())

True
C:\Users\mafal\Documents\MSc Data Science\2nd Year\Thesis\Practice\GenChordProg\data\raw\irealpro\JazzStandards.json


In [13]:
# Load and inspect the raw json file
with open(RAW_JSON_PATH, "r", encoding="utf-8") as file:
    songs = json.load(file)

print(type(songs))
print(f"Number of songs: {len(songs)}")

<class 'list'>
Number of songs: 1382


In [14]:
# Inspect first song
songs[0]

{'Title': '9.20 Special',
 'Composer': 'Earl Warren',
 'Rhythm': 'Medium Swing',
 'TimeSignature': '4/4',
 'Sections': [{'Label': 'A',
   'MainSegment': {'Chords': 'D9|Fm6|D9|Fm6|C|C7,B7,Bb7,A7'},
   'Endings': [{'Chords': 'D9|G7,Ab7,G7,'}, {'Chords': 'D9|G7,C6'}]},
  {'Label': 'B', 'MainSegment': {'Chords': 'C7|C7|F6|F6|D7|D7|G7|G7'}},
  {'Label': 'A',
   'MainSegment': {'Chords': 'D9|Fm6|D9|Fm6|C|C7,B7,Bb7,A7|D9|G7,C6'}}]}

In [15]:
# Inspect its metadata only
{
    key: songs[0].get(key)
    for key in ["Title", "Composer", "Key", "Rhythm", "TimeSignature"]
}

{'Title': '9.20 Special',
 'Composer': 'Earl Warren',
 'Key': None,
 'Rhythm': 'Medium Swing',
 'TimeSignature': '4/4'}

In [16]:
# Define conversion function
METADATA_COLUMNS = [
    "Title",
    "Composer",
    "Key",
    "Rhythm",
    "TimeSignature",
]

OUTPUT_COLUMNS = [
    "Title",
    "Composer",
    "Key",
    "Rhythm",
    "TimeSignature",
    "Chords",
]


def json_songs_to_chord_dataframe(json_data):
    if isinstance(json_data, dict):
        songs = [json_data]
    else:
        songs = json_data

    rows = []

    for song in songs:
        metadata = {
            column: song.get(column)
            for column in METADATA_COLUMNS
        }

        for section in song.get("Sections", []):
            segments = [section.get("MainSegment")]
            segments.extend(section.get("Endings", []))

            for segment in segments:
                if not segment:
                    continue

                chords = segment.get("Chords")

                if not chords or not chords.strip():
                    continue

                row = metadata.copy()
                row["Chords"] = chords

                rows.append(row)

    return pd.DataFrame(rows, columns=OUTPUT_COLUMNS)

In [17]:
# Convert to a dataframe
df = json_songs_to_chord_dataframe(songs)

df

,Title,Composer,Key,Rhythm,TimeSignature,Chords
0,9.20 Special,Earl Warren,None,Medium Swing,4/4,"D9|Fm6|D9|Fm6|C|C7,B7,Bb7,A7"
1,9.20 Special,Earl Warren,None,Medium Swing,4/4,"D9|G7,Ab7,G7,"
2,9.20 Special,Earl Warren,None,Medium Swing,4/4,"D9|G7,C6"
3,9.20 Special,Earl Warren,None,Medium Swing,4/4,C7|C7|F6|F6|D7|D7|G7|G7
4,9.20 Special,Earl Warren,None,Medium Swing,4/4,"D9|Fm6|D9|Fm6|C|C7,B7,Bb7,A7|D9|G7,C6"
...,...,...,...,...,...,...
5067,Zingaro (Retrato Em Branco E Preto),Antonio-Carlos Jobim,Gmin,Bossa Nova,4/4,G7b13|G7b13
5068,Zingaro (Retrato Em Branco E Preto),Antonio-Carlos Jobim,Gmin,Bossa Nova,4/4,"Gm7|Ebm7,Ebm6"
5069,Zoltan,Woody Shaw,None,Latin,4/4,Cmaj7#11|Cmaj7#11|Cmaj7#11|Cmaj7#11|Bbmaj7#11|Bbmaj7#11|Bbmaj7#11|Bbmaj7#11
5070,Zoltan,Woody Shaw,None,Latin,4/4,"Gbmaj7#11|Gbmaj7#11|Gbmaj7#11|Fmaj7#11|Ebmaj7#11|Ebmaj7#11|Ebmaj7#11,Fmaj7#11|Abmaj7#11,Bbmaj7#11"


##### Basic Checks:

In [18]:
print(f"Rows: {len(df)}")
print(f"Unique songs: {df['Title'].nunique()}")
print(f"Rows with missing Key: {df['Key'].isna().sum()}")
print(f"Rows with empty Chords: {(df['Chords'].str.strip() == '').sum()}")

Rows: 5072
Unique songs: 1382
Rows with missing Key: 892
Rows with empty Chords: 0


In [19]:
# Look at the first 20 rows:
df.head(20)

,Title,Composer,Key,Rhythm,TimeSignature,Chords
0,9.20 Special,Earl Warren,None,Medium Swing,4/4,"D9|Fm6|D9|Fm6|C|C7,B7,Bb7,A7"
1,9.20 Special,Earl Warren,None,Medium Swing,4/4,"D9|G7,Ab7,G7,"
2,9.20 Special,Earl Warren,None,Medium Swing,4/4,"D9|G7,C6"
3,9.20 Special,Earl Warren,None,Medium Swing,4/4,C7|C7|F6|F6|D7|D7|G7|G7
4,9.20 Special,Earl Warren,None,Medium Swing,4/4,"D9|Fm6|D9|Fm6|C|C7,B7,Bb7,A7|D9|G7,C6"
5,26-2,John Coltrane,F,Medium Up Swing,4/4,"Fmaj7,Ab7|Dbmaj7,E7|Amaj7,C7|Cm7,F7|Bbmaj7,Db7|Gbmaj7,A7|Dm7,G7|Gm7,C7"
6,26-2,John Coltrane,F,Medium Up Swing,4/4,"Fmaj7,Ab7|Dbmaj7,E7|Amaj7,C7|Cm7,F7|Bbmaj7,Ab7|Dbmaj7,E7|Amaj7,C7|Fmaj7"
7,26-2,John Coltrane,F,Medium Up Swing,4/4,"Cm7,F7|Em7,A7|Dmaj7,F7|Bbmaj7|Ebm7|Ab7|Dbmaj7|Gm7,C7"
8,26-2,John Coltrane,F,Medium Up Swing,4/4,"Fmaj7,Ab7|Dbmaj7,E7|Amaj7,C7|Cm7,F7|Bbmaj7,Ab7|Dbmaj7,E7|Amaj7,C7|Fmaj7"
9,52nd Street Theme,Thelonious Monk,None,Up Tempo Swing,4/4,"C,Am7|Dm7,G7|C,Am7|Dm7,G7|C,Am7|Dm7,G7|C,G7|C"


In [20]:
# Look at one specific song:
df[df["Title"] == "Alone Together"]

,Title,Composer,Key,Rhythm,TimeSignature,Chords
263,Alone Together,Arthur Schwartz,Dmin,Medium Swing,4/4,"Dm6|Em7b5,A7b9|Dm6|Em7b5,A7b9|Dm6|Am7b5,D7b9|Gm7|Gm7|Bm7,E7|Gm7,C7|Fmaj7|Em7b5,A7b9"
264,Alone Together,Arthur Schwartz,Dmin,Medium Swing,4/4,"Dmaj7|Dmaj7(Em7b5),(A7b9)"
265,Alone Together,Arthur Schwartz,Dmin,Medium Swing,4/4,Dmaj7|Dmaj7
266,Alone Together,Arthur Schwartz,Dmin,Medium Swing,4/4,"Am7b5|D7b9|Gm6|Gm6|Gm7b5|C7b9|Fmaj7|Em7b5,A7b9"
267,Alone Together,Arthur Schwartz,Dmin,Medium Swing,4/4,"Dm6|Em7b5,A7b9|Dm6|Em7b5,A7b9|Dm6,Bm7b5|Bb7,A7b9|Dm6|Em7b5,A7b9"


In [21]:
# Look at songs without a key:
df[df["Key"].isna()].head(20)

,Title,Composer,Key,Rhythm,TimeSignature,Chords
0,9.20 Special,Earl Warren,None,Medium Swing,4/4,"D9|Fm6|D9|Fm6|C|C7,B7,Bb7,A7"
1,9.20 Special,Earl Warren,None,Medium Swing,4/4,"D9|G7,Ab7,G7,"
2,9.20 Special,Earl Warren,None,Medium Swing,4/4,"D9|G7,C6"
3,9.20 Special,Earl Warren,None,Medium Swing,4/4,C7|C7|F6|F6|D7|D7|G7|G7
4,9.20 Special,Earl Warren,None,Medium Swing,4/4,"D9|Fm6|D9|Fm6|C|C7,B7,Bb7,A7|D9|G7,C6"
9,52nd Street Theme,Thelonious Monk,None,Up Tempo Swing,4/4,"C,Am7|Dm7,G7|C,Am7|Dm7,G7|C,Am7|Dm7,G7|C,G7|C"
10,52nd Street Theme,Thelonious Monk,None,Up Tempo Swing,4/4,C7|C7|F6|F6|D7|D7|G7|G7
11,52nd Street Theme,Thelonious Monk,None,Up Tempo Swing,4/4,"C,Am7|Dm7,G7|C,Am7|Dm7,G7|C,Am7|Dm7,G7|C,G7|C"
17,A Ballad,Gerry Mulligan,None,Ballad,4/4,"Dm7,G7|Cmaj7,Am7|C#m7,F#7#9|Bmaj7|Em7,A7b9#5|F#m7,B7b9"
18,A Ballad,Gerry Mulligan,None,Ballad,4/4,"Em7,A7|Dmaj7,B7b13|Em7,Gm6|F#m7,F07|Em7,A7"


In [22]:
# Look at chord-segment lengths
df["num_chord_groups"] = (
    df["Chords"]
    .str.split("|")
    .str.len()
)

df["num_chord_groups"].describe()

count    5072.000000
mean        7.819598
std         4.561937
min         1.000000
25%         5.000000
50%         8.000000
75%         8.000000
max        35.000000
Name: num_chord_groups, dtype: float64

In [23]:
# Inspect unusually long or short rows:
df.sort_values("num_chord_groups", ascending=False).head(20)

,Title,Composer,Key,Rhythm,TimeSignature,Chords,num_chord_groups
159,Ahmid-6,Pat Metheny,None,Even 8ths,4/4,"Dm7|C#m7|Dm7|C#m7|F#m7,B7|Em7,A7|G,F|Em,Dm7|Cmaj7|B7|Bbmaj7#11|Am7b5,D7b9|F#m7b5,B7b9|Em7b5,A7b9|Dm7,Em7|Fm7,Bb7|Eb9...",35
832,Captain Marvel,Chick Corea,None,Samba,4/4,Em7|Em7|Bm7|Bm7|F#m7|F#m7|Bbm7|Bbm7/Ab|Gm7b5|C7b9|Dbmaj7|Gbmaj7#11|Cm7/F|F7|Bbmaj7|Bbmaj7|Ebmaj7#11|Ebmaj7#11|Bbmaj7...,34
4729,Waltz For Debby,Bill Evans,F,Waltz,3/4,Fmaj7/A|Dm7|Gm7|C7|A7/G|D7/F#|G7/F|C7/E|F7/Eb|Bb6/D|Gm7b5/Db|C7|Am7|Dm7|Gm7|C7|Fmaj7/A|Dm7|Gm7|C7|A7/C#|D7/C|G7/B|C7...,32
1049,Dance Cadaverous,Wayne Shorter,None,Waltz,3/4,Bm/maj7|Bm/maj7|Cm/maj7|Cm/maj7|C#m7b5|C#m7b5|F#7b9|F#7b9|Bm/maj7/D|Bm/maj7/D|Am7/D|D7|Gmaj7#5|Gmaj7#5/F#|Em7|Em7/D|...,32
1102,Dear Lord,John Coltrane,D,Medium Swing,4/4,Dmaj7|Dmaj7|Dmaj7|Dmaj7|Em7|Em7|Em7|Em7|Dmaj7|Dmaj7|Dmaj7|Dmaj7|Bm|Bm/maj7|Bm7|Bm6|Bm|Bm/maj7|Bm7|Bm6|Em7|Em7|A7sus|...,32
1103,Dear Lord,John Coltrane,D,Medium Swing,4/4,Dmaj7|Dmaj7|Dmaj7|Dmaj7|Em7|Em7|Em7|Em7|Dmaj7|Dmaj7|Dmaj7|Dmaj7|Gmaj7|Gmaj7|Gmaj7|Gmaj7|Gmaj7|Gmaj7|Gmaj7|Gmaj7|F#m7...,32
4731,Waltz For Debby,Bill Evans,F,Waltz,3/4,Fmaj7/A|Dm7|Gm7|C7|A7/G|D7/F#|G7/F|C7/E|F7/Eb|Bb6/D|Gm7b5/Db|C7|Am7|Dm7|Bm7|E7|Am7|F7|Bbmaj7|A7alt|Dm7|Dm7|G7|G#07|A...,32
2316,In A Silent Way,Josef Zawinul,C#min,Latin,4/4,"E|D6/E|D6/E|Fmaj7#11/E|E|E,G#m7|Amaj7|G#m7|G#m7|Dmaj7/E|Dmaj7/E,D,C#m7,|Bm7,E7|C#m7,C/G#|A/F|A/F,Dmaj9|B/C#|B/C#,Bm6...",32
2786,Litha,Chick Corea,D,Up Tempo Swing,4/4,Em7|Em7|Em7|Em7|Em7|Em7|Em7|Em7|Bb7sus|Bb7sus|Bb7sus|Bb7sus|Bb7sus|Bb7sus|Bb7sus|Bb7sus|Am7|Am7|Am7|Am7|Am7|Am7|Am7|...,32
1048,Dance Cadaverous,Wayne Shorter,None,Waltz,3/4,Bm/maj7|Bm/maj7|Cm/maj7|Cm/maj7|C#m7b5|C#m7b5|F#7b9|F#7b9|Bm/maj7|Bm/maj7|Cm/maj7|Cm/maj7|C#m7|C#m7|F#7sus|F#7|Gmaj7...,32


In [24]:
# Look at common chord labels
all_chords = (
    df["Chords"]
    .str.split("|")
    .explode()
    .str.split(",")
    .explode()
    .str.strip()
)

all_chords = all_chords[all_chords != ""]

all_chords.value_counts().head(30)

Chords
C7        2549
Gm7       2458
G7        2226
Bb7       2201
Dm7       2012
F7        1931
Cm7       1818
Fm7       1738
Am7       1723
D7        1698
Eb7       1371
Fmaj7     1173
Ebmaj7    1109
Bbm7       954
A7         947
Em7        908
Cmaj7      902
Bbmaj7     865
Abmaj7     758
Ab7        758
F6         723
Gmaj7      621
E7         621
Db7        568
Eb6        558
C6         558
Bm7        514
Ebm7       470
Bb6        463
D7b9       415
Name: count, dtype: int64

This is useful for spotting malformed labels, unusual extensions, or chords you may want to filter later.

##### Save Dataset

In [ ]:
# Save the dataset
df.to_csv(
    "C:\\Users\\mafal\\Documents\\MSc Data Science\\2nd Year\\Thesis\\Practice\\ChordProgRLHF\\data\\raw\\irealpro\\JazzStandards.csv",
    index=False
)